
# NOTEBOOK: 01_BRONZE_PIPELINE
Description: Ingests raw data from Azure SQL (JDBC), Lakeflow, and ADLS Gen2 (CSV/JSON).

In [0]:

# STEP 1: Import Libraries & Configurations
from pyspark.sql.functions import col, collect_list, concat_ws, regexp_replace
import json

In [0]:

# STEP 2: Define Storage Paths & Connections
storage_base_path = "**/path/to/storage/"
users_file_path = f"{storage_base_path}users_data.csv"
mcc_file_path = f"{storage_base_path}mcc_codes.json"
fraud_file_path = f"{storage_base_path}train_fraud_labels.json"

In [0]:

# STEP 3: Setup Databases
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

DataFrame[]

In [0]:

# STEP 4: Ingest Transactions (Azure SQL via JDBC)

# JDBC

server = "jarvis-sql-server.database.windows.net"
database = "transactions"
username = "sqladmin"
password = "Password123!"

url = f"jdbc:sqlserver://{server}:1433;database={database};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

transactions_df = (
    spark.read
    .format("jdbc")
    .option("url", url)
    .option("dbtable", "dbo.transactions_data")
    .option("user", username)
    .option("password", password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

transactions_df.write.mode("overwrite").saveAsTable("bronze.transactions_data")

print("bronze.transactions_data count:", spark.table("bronze.transactions_data").count())

bronze.transactions_data count: 13305915


In [0]:

# STEP 5: Ingest Users (CSV from ADLS)
users_df = spark.read.option("header", "true").option("inferSchema", "true").csv(users_file_path)
users_df.write.mode("overwrite").saveAsTable("bronze.users_data")

In [0]:

# STEP 6: Ingest MCC Codes (JSON from ADLS)
mcc_text_df = spark.read.text(mcc_file_path)
mcc_text = mcc_text_df.select(regexp_replace(concat_ws("", collect_list(col("value"))), "^ *\ufeff*", "").alias("jt")).first()["jt"]
mcc_raw_df = spark.createDataFrame([(int(k), v) for k, v in json.loads(mcc_text).items()], ["mcc_code", "mcc_description"])
mcc_raw_df.write.mode("overwrite").saveAsTable("bronze.mcc_codes")

In [0]:

# STEP 7: Ingest Fraud Labels (JSON from ADLS)
fraud_text_df = spark.read.text(fraud_file_path)
fraud_text = fraud_text_df.select(regexp_replace(concat_ws("", collect_list(col("value"))), "^ *\ufeff*", "").alias("jt")).first()["jt"]
fraud_mapping = json.loads(fraud_text)["target"]
fraud_raw_df = spark.createDataFrame([(int(k), v) for k, v in fraud_mapping.items()], ["transaction_id", "fraud_label"])
fraud_raw_df.write.mode("overwrite").saveAsTable("bronze.train_fraud_labels")

In [0]:
# Step 8: cards_data validation
# This assumes Lakeflow already created bronze.cards_data

print("bronze.cards_data count:", spark.table("bronze.cards_data").count())


# Final bronze validation

print("bronze.transactions_data:", spark.table("bronze.transactions_data").count())
print("bronze.cards_data:", spark.table("bronze.cards_data").count())
print("bronze.users_data:", spark.table("bronze.users_data").count())
print("bronze.mcc_codes:", spark.table("bronze.mcc_codes").count())
print("bronze.train_fraud_labels:", spark.table("bronze.train_fraud_labels").count())

bronze.cards_data count: 6146
bronze.transactions_data: 13305915
bronze.cards_data: 6146
bronze.users_data: 2000
bronze.mcc_codes: 109
bronze.train_fraud_labels: 8914963
